# Business Entity Resolution — Amazon ML Challenge 2026

**Task.** For every business in Source 1 (a de-duplicated reference list), find all records in
Source 2 and Source 3 (noisy copies) that describe the same business. An S1 record can have
0, 1 or many matches.

**Metric.** Macro F0.5 per S1 record (precision counts twice as much as recall). An S1 with no
true matches scores 1 only if we predict nothing for it.

**Constraints.** Only the provided data can be used (no external look-ups, APIs or geocoding).
Any model must be MIT/Apache licensed and at most 8B parameters. The test set contains
**France**, which does not appear in the training data.

## How the solution works

| Step | What it does | Module |
|---|---|---|
| 1. Normalization | Cleans names and addresses: casing, accents, legal forms, street-type abbreviations, learned Indic transliteration, state and region names | `src/normalize.py`, `src/data.py` |
| 2. Blocking | Fast TF-IDF inverted-index search that finds candidate pairs through name, address, combined and reverse channels | `src/blocking.py` |
| 3. Stage-1 model | LightGBM on pair features: string similarity, token alignment, house numbers, legal form, raw-name style, name frequency | `src/features.py`, `src/model.py` |
| 4. Stage-2 model | LightGBM that sees every candidate of an S1 at once: agreement between siblings and S2/S3 slots | `src/stage2.py` |
| 5. Decision rule | Each S2/S3 record goes only to its best S1. Keep links with p ≥ 0.86, plus each S1's best candidate if p ≥ 0.55 | this notebook |

No LLMs, sentence embeddings or external data are used. Everything runs on a 16 GB laptop CPU.

## 0. Setup

This notebook can be run from the repository root or from `code/business_entity_resolution/`.
Data is read from `dataset/`, outputs go to `output/` and caches to `work/`.

`RUN_FULL = False` runs only the quick demonstration cells (seconds to minutes).
Set it to `True` to re-run the full pipeline: training takes about 1.5 h and prediction about 2.5 h
on a 16 GB machine. Run one heavy job at a time.

In [ ]:
import sys, json, subprocess, time
from pathlib import Path

# Resolve PKG and ROOT dynamically whether executed from root or from code/business_entity_resolution
cwd = Path.cwd()
if (cwd / 'code' / 'business_entity_resolution').exists():
    ROOT = cwd
    PKG = cwd / 'code' / 'business_entity_resolution'
elif (cwd / 'src').exists() and (cwd / 'artifacts').exists():
    PKG = cwd
    ROOT = cwd.parent.parent
elif (cwd.parent / 'code' / 'business_entity_resolution').exists():
    ROOT = cwd.parent
    PKG = cwd.parent / 'code' / 'business_entity_resolution'
else:
    PKG = cwd
    ROOT = cwd.parent.parent

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))

DATA, WORK, OUTPUT = ROOT / 'dataset', ROOT / 'work', ROOT / 'output'

import numpy as np
import polars as pl
import lightgbm as lgb

from src.config import TRAIN_DIR, TEST_DIR, ARTIFACT_DIR, RANDOM_SEED
from src.data import read_tsv, read_ground_truth, gt_pairs, get_transliterator
from src.normalize import normalize_frame
from src.blocking import BlockingConfig
from src.model import LGB_PARAMS, macro_f05
from src.stage2 import STAGE2_COLS

RUN_FULL = False                      # True = re-run training / prediction (hours)

def run(cmd):
    # run a pipeline command from the package folder and stream its log
    print('>>', cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=PKG)

pl.Config.set_tbl_rows(20); pl.Config.set_tbl_width_chars(160)
print('artifacts:', sorted(p.name for p in ARTIFACT_DIR.iterdir()))
print('ROOT:', ROOT)
print('PKG:', PKG)


## 1. The data

Each source file has `entity_id, business_name, business_address, country`.
`train_ground_truth.tsv` maps each S1 id to its matching S2/S3 ids.

In [ ]:
train_s1 = read_tsv(TRAIN_DIR / "train_source1.tsv")
test_s1 = read_tsv(TEST_DIR / "test_source1.tsv")
print("train S1:", train_s1.height, " test S1:", test_s1.height)
print(test_s1.group_by("country").len().sort("country"))
test_s1.filter(pl.col("country") == "France").head(5)

### What the ground truth shows

* Every S1 appears in the ground truth. About **5.6 %** have no match at all; these are singletons
  that only score if we predict nothing for them.
* On average an S1 has **3.46 matches** (1.67 in S2, 1.79 in S3). This distribution is **identical for US
  and India**, which means the data generator does not depend on country. That lets us judge
  France: if we predict far fewer links per S1 there, we are missing matches.
* Records without an address are almost always real copies (about 98 % have an owner). Records with an
  address include many deliberate **look-alikes**: the same name plus a word like
  *Holding / Group / Participations*, usually at a different house number on the same street.

In [ ]:
gt = read_ground_truth(TRAIN_DIR)
cnt = (gt.with_columns(pl.col("matched_entity_ids").fill_null("").str.split(",")
                       .list.eval(pl.element().filter(pl.element() != "")).list.len().alias("n"))
         .join(train_s1.select(pl.col("entity_id").alias("source1_entity_id"), "country"), on="source1_entity_id"))
cnt.group_by("country").agg(pl.col("n").mean().alias("links_per_S1"),
                            (pl.col("n") == 0).mean().alias("share_no_match"),
                            (pl.col("n") == 1).mean().alias("share_1"),
                            (pl.col("n") >= 5).mean().alias("share_5plus"))

## 2. Normalization (`src/normalize.py`)

What happens to names:
* lower-casing, removing accents (`unidecode`) and punctuation;
* a **learned transliteration map** (1,347 tokens) for Devanagari and other Indic scripts, learned only
  from training pairs, e.g. मार्केटिंग → marketing;
* legal forms are canonicalized (`private limited → pvt ltd`, `compagnie/cie → co`, French forms
  SARL, SAS, EURL, SCI, EI…). `name_core` is the name without legal forms or stop words;
  `name_full` keeps them.

What happens to addresses:
* street types are abbreviated (`street → st`, `R. → rue`, `Bd → blvd`, …) and ordinals unified;
* India: state names and codes become one token, and historical city names are mapped
  (Bombay → Mumbai…). A learned map handles state names written in native script;
* France: department names map to their region, and `N°`, `bis`, `ter` are removed.

In [ ]:
tr = get_transliterator(TRAIN_DIR)        # learned map, cached in artifacts/translit_map.json
demo = pl.concat([test_s1.filter(pl.col("country") == c).head(3) for c in ("US", "India", "France")])
normalize_frame(demo, tr).select("business_name", "name_core", "name_full", "business_address", "addr")

## 3. Blocking (`src/blocking.py`)

Comparing 1.7M × 10M records directly is impossible, so each S1 gets a short candidate list from
a numba-parallel **TF-IDF inverted index**. There are four channels:

| Channel | Index terms | Why |
|---|---|---|
| name | word tokens + character 4-grams | typos and word re-ordering |
| addr | address tokens + bigrams | renamed businesses at the same address |
| comb | name + address | the strongest single channel |
| **reverse** | each S2/S3 record searches for *its* S1 | records whose S1 is crowded out of the forward top-K |

Records without an address get a deeper reverse search (K = 20, since their name is the only clue).
Candidates are capped per channel. This gives about 45 candidates per S1 (77.8M pairs on test) with
**≈ 99.8 % recall for US records with an address**. Most of the recall that is lost is on address-less
records whose name is shared by many S1 businesses.

In [ ]:
BlockingConfig()

In [ ]:
# Full blocking + features + models for training (about 1.5 h). The cached candidates and features
# live in ../../work/ and are reused when present.
if RUN_FULL:
    run("python -m src.pipeline --mode train --n-train 150000 --n-val 30000")
    run("python -m src.bigtrain 2 2")      # v10: stage 1 on 150k + 600k extra S1 (hard negatives kept)

### Test-like training

The test has about 2.3 unmatched S2/S3 records per S1, versus about 1.2 in train. To get closer to
that density, training removes 19 % of the training S1 records (chosen by hash), which turns their
S2/S3 records into unmatched look-alikes. Then 150k S1 are used for fitting and **30k S1 are held out**.
The held-out set is the same across every model version, so scores can be compared.

## 4. Stage-1 pair model (`src/features.py`)

86 features for each (S1, candidate) pair:

* **String similarity** (rapidfuzz): ratio, token-set, Jaro-Winkler and partial ratio on names,
  addresses and combined text.
* **Token alignment**: an extra word in the candidate can be a typo, generic noise the generator adds
  to real copies (*services, center, trust…*; learned from training pairs), or a real different word.
  A real different word is the signature of a look-alike.
* **House numbers**: equal, truncated or padded (noise), one digit different, or a real conflict (look-alike).
* **Raw name and legal form**: legal form agreement (LLC vs Inc, SARL vs SAS), raw-string ratio, and how
  common the raw name is. These features gave the biggest leaderboard gain.
* **Ambiguity**: how many S1 records share the name, candidate counts and blocking ranks/scores.

The model is LightGBM (binary cross-entropy, early stopping on the held-out S1).

In [ ]:
print(LGB_PARAMS)
stage1 = lgb.Booster(model_file=str(ARTIFACT_DIR / "lgb_matcher.txt"))
imp = sorted(zip(stage1.feature_importance("gain"), stage1.feature_name()), reverse=True)
pl.DataFrame({"feature": [n for _, n in imp[:25]], "gain": [round(g) for g, _ in imp[:25]]})

## 5. Stage-2 group model (`src/stage2.py`)

A pair is judged better with the S1's other candidates in view. Stage 2 is a second LightGBM trained
on **out-of-fold** stage-1 probabilities, so it never sees a probability the model produced for its own
training rows. Its extra features:

* **Consensus**: do the S1's confident candidates agree on house number and street (`hn_share`, `addr_share`)?
* **Siblings**: how similar is this candidate to the S1's confident candidates, and to the ones it rejected?
* **Source slots**: whether the S1 already has a confident candidate from the same source (S2 or S3) or from the other one.
* **Competition**: how this S1 compares with the other S1 records that the candidate's reverse search found
  (`rev_rel2`, `rev_gap2`).

In [ ]:
print(len(STAGE2_COLS), "stage-2 features")
print(STAGE2_COLS)

## 6. The metric and the decision rule

`macro_f05` below is the exact challenge metric (it is identical to `src/model.py`).
We verified it against the official example.

The decision rule turns probabilities into links:
1. **Exclusivity**: each S2/S3 record belongs to one business, so it is kept only for the S1 with its highest p.
2. Keep every remaining link with **p ≥ 0.86**. Records **without an address** (97.7 % real copies in train) are decided by
   how dominant their best S1 is: share = p / (sum of p over the record's S1 candidates); keep if (p ≥ 0.60, share ≥ 0.70),
   (p ≥ 0.40, share ≥ 0.90) or (p ≥ 0.30, share ≥ 0.95). Two plausible owners make a coin flip, which loses at F0.5.
3. Also keep each S1's single best candidate if **p ≥ 0.55**. This recovers S1s that would otherwise get
   no prediction at all, which is costly for S1s that do have matches.

The thresholds were first tuned on the held-out S1 and then checked on the leaderboard
(extras 0.78 / 0.82 / 0.86 and top-1 0.45 / 0.55 / 0.65); 0.86 / 0.55 scored best.

In [ ]:
def macro_f05_demo(pred, truth, s1_ids):
    # per-S1 F0.5 = 1.25*P*R / (0.25*P + R); an S1 with no true matches scores 1 only if nothing is predicted
    return macro_f05(pred, truth, s1_ids)

truth = pl.DataFrame({"s1_id": ["A", "A", "B"], "t_id": ["x", "y", "z"]})
pred = pl.DataFrame({"s1_id": ["A", "C"], "t_id": ["x", "w"]})            # A: P=1 R=.5 ; B: missed ; C: false positive on a singleton
print(macro_f05_demo(pred, truth, pl.Series(["A", "B", "C", "D"])))       # D: correct empty prediction -> 1


def final_rule(scores: pl.DataFrame, noaddr: pl.DataFrame, extras=0.86, top1=0.55,
               tiers=((0.60, 0.70), (0.40, 0.90), (0.30, 0.95))) -> pl.DataFrame:
    # scores: (s1_idx, t_idx, p) after stage 2; noaddr: (t_idx, noaddr)  ->  selected (s1_idx, t_idx)
    tot = scores.group_by("t_idx").agg(pl.col("p").sum().alias("p_sum"))
    d = scores.filter(pl.col("p") == pl.col("p").max().over("t_idx"))                  # exclusivity
    d = d.with_columns(pl.col("p").rank("ordinal", descending=True).over("s1_idx").alias("rk"))
    d = d.join(tot, on="t_idx").with_columns((pl.col("p") / pl.col("p_sum")).alias("share")).join(noaddr, on="t_idx")
    na_ok = pl.lit(False)
    for p_min, share_min in tiers:                                                     # address-less: dominant owner only
        na_ok = na_ok | ((pl.col("p") >= p_min) & (pl.col("share") >= share_min))
    extra = pl.when(pl.col("noaddr")).then(na_ok).otherwise(pl.col("p") >= extras)
    return d.filter(extra | ((pl.col("rk") == 1) & (pl.col("p") >= top1))).select("s1_idx", "t_idx")

## 7. Prediction on the test set

`--mode predict` runs these steps:
1. Normalizes and blocks all 1.73M test S1 records against 9.97M S2/S3 records and writes
   `output/candidate_pairs.tsv`.
2. Scores the 77.8M pairs with stage 1, streamed in 2M-pair chunks to stay within 16 GB.
3. Applies stage 2.
4. Re-scores French pairs with the French self-trained stage 1 (`artifacts/lgb_matcher_unseen.txt`) and the French
   noise words (section 8), and saves `work/test_final_scores.parquet`.
5. Applies the final rule from `artifacts/rule.json`, keeps at most 5 S2 and 6 S3 links per S1 (never exceeded in
   train), adds exact duplicates of already selected records, and writes `output/matching_results.tsv`.

The cell below re-applies the same rule from the saved scores, so different thresholds can be tried in seconds.

In [ ]:
if RUN_FULL:
    run("python -m src.pipeline --mode predict")

scores_path = WORK / "test_final_scores.parquet"
if scores_path.exists():
    from src.pipeline import write_id_lists
    s1_ids = pl.read_parquet(WORK / "test_s1_norm.parquet", columns=["entity_id"])["entity_id"]
    t_ids = pl.concat([pl.read_parquet(WORK / f"test_s{i}_norm.parquet", columns=["entity_id"]) for i in (2, 3)])["entity_id"]
    from src.pipeline import quota_and_duplicates
    scores = pl.read_parquet(scores_path)
    tg_raw = pl.concat([pl.read_parquet(WORK / f"test_s{i}_norm.parquet", columns=["entity_id", "addr", "business_name", "business_address"]) for i in (2, 3)])
    noaddr = tg_raw.select(pl.int_range(pl.len(), dtype=pl.Int32).alias("t_idx"),
                           (pl.col("addr").is_null() | (pl.col("addr").str.strip_chars() == "")).alias("noaddr"))
    sel = quota_and_duplicates(final_rule(scores, noaddr), scores, tg_raw)
    print(f"{sel.height:,} links for {sel['s1_idx'].n_unique():,} S1")
    if RUN_FULL:
        write_id_lists(sel, s1_ids, t_ids, OUTPUT / "matching_results.tsv", "matched_entity_ids")

## 8. France: a country with no training data

Everything learned from US and India has to transfer to France. What we do:
* **Language-independent features.** The model never sees the country name as a feature. Noise words,
  house-number relations and legal forms are handled generically.
* **French normalization.** Street-type abbreviations (R., Av., Bd, Imp., Chem.), department → region,
  removing `N°`, `bis`, `ter`, and French legal forms (SARL, SAS, SASU, EURL, SCI, EI, SELARL, GIE, EARL).
* **Link-count check.** Because the generator does not depend on country, France should have about 3.46 links
  per S1. We predict 3.16–3.20, so France is under-linked. That makes it the biggest remaining source of loss
  (France scores ≈ 0.957 against ≈ 0.985 for US/India on the held-out estimate).
* **French noise words.** In French candidates at the *same* house number and street, the extra
  words *France, Groupe, Développement, Associés* appear about 30× more often than the look-alike rate for other
  suffix words. So the generator uses them as noise on real copies, the way *services / center* are used in the
  US. They are passed to the token-alignment features as country-specific noise words (`COUNTRY_NOISE`).
  The same words are also used to build look-alikes, but look-alikes almost always get a *different* house number.
  So the re-scored probability is used only where the house numbers agree or the record has no address; every other
  French pair keeps its original score (`work/frnz.py`, `work/frnz_merge.py`). On the US/India held-out set, the matching
  pattern (same house number, one word dropped, a noise word added) is a true match 93 % of the time.
* **Self-training (three rounds).** Confident French predictions (p ≥ 0.95 as matches; p ≤ 0.02 as non-matches, sampled and
  re-weighted to the real base rate) continue training the stage-1 model for 400 rounds, with 3M US/India rows as
  an anchor and the noise words switched on. The adapted model re-scores every French pair; US/India keep their scores.
  Round 2 takes its pseudo-labels from round 1 and adds *Fils* ("& Fils") as a fifth noise word.
* **Rules we tested and rejected.** Dropping French links whose house number differs on the same street
  (proposed by an external audit) lost 0.00042 on the leaderboard. On US/India held-out, such links are 99.5 % correct.

In [ ]:
if RUN_FULL:
    # pseudo-labels from the guarded noise-word scores of the first prediction run
    import shutil
    shutil.copy(WORK / "test_final_scores.parquet", WORK / "test_final_scores_frnz_merge.parquet")
    run("python -m src.selftrain_fr v12fr frnz_merge")        # round 1 (about 30 min)
    run("python -m src.selftrain_fr v13fr v12fr fils")      # round 2
    run("python -m src.selftrain_fr v14fr v13fr fils")      # round 3
    shutil.copy(WORK / "artifacts_v14fr_stage1.txt", ARTIFACT_DIR / "lgb_matcher_unseen.txt")
    run("python -m src.pipeline --mode predict")               # France re-scored by the adapted model

## 9. Validation and packaging

In [ ]:
if RUN_FULL:
    run("python ../../utils/validate_submission.py --matching ../../output/matching_results.tsv "
        "--candidate ../../output/candidate_pairs.tsv --test-dir ../../dataset/test --check-ids")
    run("python ../../utils/package_submission.py --team-name TEAM_NAME")

## 10. Results

| Version | What changed | Leaderboard |
|---|---|---|
| v5 / v7 | earlier iterations of the same two-stage design | 0.9694 / 0.9776 |
| v8 | test-like training, stage 2 with sibling and slot features (held-out 0.98499) | 0.978406 |
| v9 | stage 1 trained on 3× more businesses (300k extra S1, hard negatives kept) | 0.979268 |
| v9 + rule | exclusivity, extras 0.82, top-1 0.55 | 0.980496 |
| v10h + rule | stage 1 with 600k extra S1 | 0.980742 |
| v10h, extras 0.86 / top-1 0.55 | stricter threshold for extra links | 0.980833 |
| v10h + French noise words (house-number guard) | France, Groupe, Développement, Associés treated as noise where house numbers agree | 0.982206 |
| + Fils, S2/S3 quota, exact duplicates | fifth noise word and two structural checks | 0.982631 |
| + French self-training round 1 | confident French pseudo-labels continue stage 1 | 0.982798 |
| round 1 + quota + duplicates | | 0.982802 |
| round 2 (+ Fils) + quota + duplicates | second self-training round | 0.983159 |
| + address-less extras 0.60 | 0.70 / 0.60 / 0.50 / 0.40 → 0.983552 / 0.983579 / 0.983512 / 0.983512 | 0.983579 |
| self-training round 3 + address-less 0.60 | third round, pseudo-labels from round 2 | 0.983732 |
| + share-based address-less rule | keep address-less records only when one S1 dominates | 0.98396 / 0.983969 |
| **Final** | share tiers (0.60, 0.70), (0.40, 0.90), (0.30, 0.95) | **0.983987** |

**Final submission: 0.983987.**

On the 30k held-out S1 (US/India only), the final rule scores 0.98492 with precision ≈ 99.8 % and recall ≈ 98.3 %
on records with an address, and ≈ 49 % recall on records without one.

### What limits the score
* **Records without an address whose name is shared by several S1 businesses (chains).** Nothing in the data says
  which branch owns such a record, so the best choice is often to leave it out. This is a large part of the
  remaining US/India loss, and it cannot be recovered from the data.
* **France.** France still has fewer links per S1 than the generator produces. Its look-alikes (same name with a
  different legal form and a nearby house number) are hard to tell apart from real copies without labels.
* **Blocking.** Recall is 99.8 % for US records with an address and 98.8 % for India. It is only about 85 % for
  address-less records in large name families.